In [35]:
import numpy as np
import scipy.sparse as sp
import random

Arquivos necessários para o teste

In [36]:
GID_1 = np.load('arquivos/GID_1.npy')
GID_0 = np.load('arquivos/GID_0.npy')
lines_operator, columns_operator, values_operator = np.load('arquivos/op_132000_576.npy')
dual = np.load('arquivos/DUAL_1.npy')

lines_operator = lines_operator.astype(int)
columns_operator = columns_operator.astype(int)

# Funciona apenas para o caso de 2 níveis
primal_vertex = np.empty(GID_1.shape[0], dtype=int)
primal_vertex[GID_1[(dual == 3)]] = np.arange(len(dual))[dual == 3]

Teste para apenas 2 níveis: fino e grosso (0 e 1)

In [37]:
class NUADM_ID:
    def __init__(self, GID0, GID1):
        self.mapping = GID1
        self.index_atual = 0
        self.ids = None # 
        self.id_NUADM = None
        self.num_volumes = len(GID0)
        self.get_index_vector = GID0
        self.operator_index = np.arange(self.num_volumes)

    def generate_ids(self, level_vector):
            # inicializando vetor de ids NU-ADM
            self.id_NUADM = np.full(self.num_volumes, -1)
            self.ids = np.full(len(self.mapping), -1)
            print('ids initialized')
            
            index_0 = self.get_index_vector[level_vector == 0]
            self.id_NUADM[index_0] = np.arange(len(index_0))
            self.index_atual += len(index_0)

            index_1 = self.get_index_vector[level_vector == 1]
            grossa_index = np.unique(self.mapping[index_1])
            self.ids[grossa_index] = np.arange(self.index_atual, self.index_atual + len(grossa_index))
            self.id_NUADM[index_1] = self.ids[self.mapping[index_1]]
            self.index_atual += len(grossa_index)

    # Gera o operador de prolongamento entre dois níveis (Apenas 2) - Level vector tem que ser 
    def generate_operator(self, lines_operator, columns_operator, values_operator, level_vector, primal_vertex):
        ids_0 = self.operator_index[level_vector == 0]

        lines_0 = ids_0
        columns_0 = self.id_NUADM[ids_0]
        values_0 = np.ones_like(ids_0)

        mask = level_vector[lines_operator] == 1
        lines_1 = lines_operator[mask]
        columns_1 = self.id_NUADM[primal_vertex[columns_operator[mask]]]
        values_1 = values_operator[mask]

        new_lines = np.concatenate((lines_0, lines_1))
        new_columns = np.concatenate((columns_0, columns_1))
        new_values = np.concatenate((values_0, values_1))

        return new_lines, new_columns, new_values

In [38]:
def generate_level_vector(length):
    print('Generating level_vector')
    vector = []
    for _ in range(length):
        vector.append(random.randint(0, 1))
    vector = np.array(vector)
    return vector

In [44]:
level_vector = generate_level_vector(len(GID_0))
print('level vector:', level_vector[:20])

Generating level_vector
level vector: [1 0 1 0 1 1 1 0 0 0 1 0 1 1 1 1 0 0 1 1]


In [45]:
sla = NUADM_ID(GID_0, GID_1)
sla.generate_ids(level_vector)
print('Calculado', sla.id_NUADM[:20])
print(np.bincount(level_vector))

ids initialized
Calculado [65957     0 65957     1 65957 65958 65958     2     3     4 65957     5
 65957 65957 65957 65958     6     7 65958 65958]
[65957 66043]


In [41]:
# aux = np.arange(len(level_vector))
# def generate_operator(lines_operator, columns_operator, values_operator, vector_ids, vector_level, primal_vertex, aux):
#     # 1. Separar índices por nível
#     ids_0 = aux[vector_level == 0]

#     # --- Parte 1: Nível 0 (direto) ---
#     lines_0 = ids_0
#     columns_0 = vector_ids[ids_0]
#     values_0 = np.ones_like(ids_0)

#     # --- Parte 2: Nível 1 (indireto via operador) ---
#     # mask = np.isin(lines_operator, ids_1)
#     mask = vector_level[lines_operator] == 1

#     lines_1 = lines_operator[mask]
#     columns_1 = vector_ids[primal_vertex[columns_operator[mask]]]
#     values_1 = values_operator[mask]

#     # --- Junta tudo ---
#     new_lines = np.concatenate([lines_0, lines_1])
#     new_columns = np.concatenate([columns_0, columns_1])
#     new_values = np.concatenate([values_0, values_1])

#     return new_lines, new_columns, new_values

In [46]:
l, c, d = sla.generate_operator(lines_operator, columns_operator, values_operator, level_vector, primal_vertex)

In [47]:
import numpy as np
import gstools as gs
import pyvista as pv
import scipy.sparse as sp

def get_grid():
    ordem=[2,1,0]
    nb=np.array([nx, ny, nz])[ordem]+1
    lb=np.array([hx,hy,hz])[ordem]
    sp=np.array([0,0,0])
    grid = pv.ImageData()
    grid.dimensions = nb
    grid.origin = sp  # The bottom left corner of the data set
    grid.spacing = lb  # These are the cell sizes along each axis
    return grid
# Dimensões SPE10
# nx, ny, nz = 60, 220, 85
# Lx, Ly, Lz = 1200.0, 2200.0, 850.0
nx, ny, nz = 60, 220, 10
Lx, Ly, Lz = 1200.0, 2200.0, 850.0
hx, hy, hz = Lx/nx, Ly/ny, Lz/nz

x = np.linspace(hx/2, Lx - hx/2, nx)
y = np.linspace(hy/2, Ly - hy/2, ny)
z = np.linspace(hz/2, Lz - hz/2, nz)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Modelo geoestatístico
model = gs.Exponential(
    dim=3,
    var=1.0,
    len_scale=[300, 300, 3],
    anis=[1.0, 1.0, 0.05],
)

srf = gs.SRF(model, seed=42)
log_k = srf((X, Y, Z))
log_k = 6 * (log_k - np.mean(log_k)) / np.std(log_k)
k = np.exp(log_k)  # mD

# Salvar como .txt
k_flat = k.flatten(order='F')
grid=get_grid()
centroids=np.array([X.flatten(),Y.flatten(),Z.flatten()]).T

# grid.cell_data["ks"] = np.log10(k_flat)
l0,c0,d0=np.load('arquivos/op_132000_576.npy')
v=0
op_ms=sp.csr_matrix((d0, (l0.astype(int), c0.astype(int))), shape = (int(l0.max()+1), int(c0.max()+1)), dtype=np.float32)
FBms=op_ms[:,v].toarray().transpose()[0]

grid.cell_data["OPms_"+str(v)] = FBms
grid.cell_data["DUAL"] = dual
grid.cell_data["levels"] = level_vector

new_op=sp.csr_matrix((d, (l.astype(int), c.astype(int))), shape = (int(l.max()+1), int(c.max()+1)), dtype=np.float32)
FB=new_op[:,sla.id_NUADM[v]].toarray().transpose()[0]
grid.cell_data["NEW_OP_"+str(v)] = FB


# grid.cell_data["ks"] = np.log10(k_flat)



grid.save('arquivos/results/gs_spe10.vtk') #ativar
# import pdb; pdb.set_trace()
# np.savetxt("permeabilidade_spe10_geoestatistica.txt", k_flat, fmt="%.6e")
# import pdb; pdb.set_trace()
